# Scaling LLM inference with Ray Serve LLM: routing, disaggregation, and MoE

© 2026, Anyscale. All Rights Reserved

Once one replica is fast, the hard problems move to the fleet:

- **Which replica** should serve a request?
- **Should one replica** handle both its prefill and decode phases?
- **What changes** when you scale out a mixture-of-experts (MoE) model instead of a dense one?

<div class="alert alert-block alert-info">
<b>Roadmap for this notebook</b>
<ol>
    <li>KV-cache-aware routing.</li>
    <li>Prefill/decode disaggregation.</li>
    <li>Mixture-of-experts at scale.</li>
    <li>Tier-2 multi-node production configs.</li>
</ol>
</div>

**Imports**

In [ ]:
from openai import OpenAI
from ray import serve
from ray.serve.config import RequestRouterConfig
from ray.serve.llm import (
    LLMConfig,
    build_pd_openai_app,    # build_pd_openai_app is alpha
    build_dp_openai_app,    # build_dp_openai_app is alpha
)
from ray.serve.llm.request_router import PrefixCacheAffinityRouter

Every config below loads the same Tier-1 model from a public S3 mirror rather than the Hugging Face Hub, so a room full of replicas is not pulling the same weights over the internet.

In [ ]:
MODEL_SOURCE = "s3://anyscale-public-materials-use2/models/Qwen/Qwen2.5-0.5B-Instruct"
MODEL_ENV = dict(env_vars={"AWS_REGION": "us-east-2"})   # the region this bucket lives in

Pairing it with `load_format="runai_streamer"` in `engine_kwargs` lets vLLM's Run:ai Model Streamer read the safetensors straight from the bucket into GPU memory, with no separate download to local disk.

## 1. KV-cache-aware routing

Routing across replicas is a framework job. Each engine already keeps a per-replica **key-value (KV) cache** of the prefixes it has served, but the default load-balancing router throws that cross-request reuse away.

This section names two different affinity problems, shows the one router that ships as a runnable API, and measures the payoff.

### 1.1 The routing problem

Serve's default router is **Power of Two Choices**: sample two replicas, ask each its in-flight count, and send the request to the less-loaded one. It balances *load* but is blind to the *KV cache*.

Under load-only routing the same colored prefix scatters across both replicas, and each one redoes the full prefill: a cache miss on every replica.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/routing_kv_blind.png" loading="lazy" width="860">

- **Power of Two is KV-blind**
    - It picks the shorter queue: good for load.
    - Indifferent to which replica already holds a matching prefix.
- **The cache is per-replica**
    - Each replica keeps its own prefix cache.
    - Two requests sharing a 2,000-token system prompt, landing on different replicas, each pay a full prefill of it.
    - A cache miss on both.
- **This is a routing problem, not an engine problem**
    - The engine's prefix caching works fine.
    - The fleet is just not steering a matching prefix to the replica that holds it.

### 1.2 Session-affinity routing

Before the runnable fix, name a *different* affinity problem so the two never get confused. Session affinity pins one caller's whole conversation to one replica so its growing KV cache stays warm across turns.

The same session id hashes to the same ring position on every turn, so each caller's conversation sticks to one replica and reuses a warm KV cache.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/nb5_session_affinity_ring_v2.png" loading="lazy" width="900">

- **Prefix affinity vs session affinity**
    - Prefix affinity reuses a SHARED prefix across DIFFERENT callers (section 1.3).
    - Session affinity pins ONE caller's whole conversation to ONE replica so its growing KV is reused turn over turn.
    - Different problems, different routers.
- **The router**
    - `ConsistentHashRouter` hashes a request's session id onto a consistent-hash ring so the same session lands on the same replica.
- **How the id arrives**
    - The session id is read from the request header.
    - A request with no session id falls back to a fresh per-request id, so it spreads uniformly: no affinity.
- **Attach it** through `RequestRouterConfig`, with two consistent-hash tunables.

```python
from ray.serve.config import RequestRouterConfig
from ray.serve.experimental.consistent_hash_router import ConsistentHashRouter   # NOT ray.serve.llm.request_router

deployment_config = dict(
    request_router_config=RequestRouterConfig(
        request_router_class=ConsistentHashRouter,
        request_router_kwargs=dict(
            num_virtual_nodes=100,      # ring resolution per replica (default 100)
            num_fallback_replicas=2,    # clockwise successors tried if the primary rejects (default 2)
        ),
    ),
)
# Client sends the session id on the x-session-id header; same session_id -> same ring
# position -> same replica -> warm multi-turn KV cache.
```

<div class="alert alert-block alert-warning">
<code>ConsistentHashRouter</code> lives under <code>ray.serve.experimental</code>, so treat its import path and kwargs as unstable even though it imports fine. Do not confuse it with <code>PrefixCacheAffinityRouter</code>: session affinity pins a caller, prefix affinity reuses a shared prefix.
</div>

### 1.3 Prefix-aware routing

The runnable fix for section 1.1: a prefix-tree actor tracks which replica recently served which prefix, and routing follows it.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/prefix_affinity_router.png" loading="lazy" width="960">

`PrefixCacheAffinityRouter` (from `ray.serve.llm.request_router`) routes a shared-prefix request to the replica most likely to already hold its KV, maximizing prefix-cache hits and cutting time to first token (TTFT). It falls back to Power of Two when load is imbalanced or matches are weak.

- **Requires `enable_prefix_caching=True`**
    - With no per-replica cache there is nothing for affinity to win.
    - The router is a no-op without it.
- **Two tunables**
    - `match_rate_threshold` = the minimum prefix-match fraction before affinity overrides load.
    - `imbalanced_threshold` = how skewed replica load may get before the router reverts to Power of Two.
- **Fallback keeps it safe**
    - A hot shared prefix cannot pin all traffic to one replica.
    - Skew past `imbalanced_threshold`, or matches below `match_rate_threshold`, and routing reverts to Power of Two.

In [ ]:
def routing_config(use_router: bool, n_replicas: int = 2) -> LLMConfig:
    """One model on n_replicas, with prefix-affinity routing optionally enabled."""
    deployment_config = dict(
        autoscaling_config=dict(min_replicas=n_replicas, max_replicas=n_replicas),
    )
    if use_router:
        deployment_config["request_router_config"] = RequestRouterConfig(
            request_router_class=PrefixCacheAffinityRouter,        # ray.serve.llm.request_router
            request_router_kwargs=dict(match_rate_threshold=0.1),  # min prefix-match fraction to prefer affinity
        )
    return LLMConfig(
        model_loading_config=dict(model_id="qwen-0.5b", model_source=MODEL_SOURCE),
        deployment_config=deployment_config,
        runtime_env=MODEL_ENV,
        engine_kwargs=dict(
            load_format="runai_streamer",   # required to read the s3:// source
            max_model_len=4096,
            enforce_eager=True,
            enable_prefix_caching=True,   # REQUIRED: the router is a no-op without a prefix cache to steer toward
        ),
    )

Note: the router is a **no-op unless `enable_prefix_caching=True`**. The defaults are `match_rate_threshold=0.1` and `imbalanced_threshold=inf` (affinity always wins until you cap how skewed load may get); the exact values can shift across releases, so treat them by name.

### 1.4 What the routing actually buys

The payoff grows with the fleet, and that is the whole argument. With one replica there is nothing to route between, so both policies are identical. Every replica you add gives load-only routing one more place to scatter a prefix, and one more chance to miss.

Anyscale benchmarked the two routers on `DeepSeek-R1-Distill-Qwen-32B` across 64 L4 GPUs, scaling from 1 to 16 replicas. Every request carried a 512-token prefix drawn from a pool of 32 per replica, plus a 128-token suffix, at 32 concurrent requests per replica.

| replicas | hit rate, load-only | hit rate, prefix-aware | TTFT p50, load-only | TTFT p50, prefix-aware |
|---:|---:|---:|---:|---:|
| 1 | 75% | 75% | 366 ms | 362 ms |
| 2 | 62% | 74% | 516 ms | 320 ms |
| 4 | 37% | 75% | 620 ms | 319 ms |
| 8 | 19% | 75% | 668 ms | 319 ms |
| 16 | 10% | 78% | 822 ms | 319 ms |

- **The prefix-aware line is flat**
    - The hit rate holds near 75% and TTFT holds near 319 ms at every fleet size.
    - Each prefix keeps landing on the replica that already holds it, so scaling out costs nothing in reuse.
- **The load-only line degrades with every replica added**
    - The hit rate falls 75% to 10%; TTFT more than doubles, 366 ms to 822 ms.
    - Same requests, same engines, same caches. Only the placement changed.
- **What degrades it is dilution**
    - Power of Two spreads a prefix's requests across all N replicas, so each one holds a shrinking share of it.
    - The wider the fleet, the more often a request meets a replica that has never seen its prefix.
- **The headline at 16 replicas**
    - About 60% off TTFT and more than 40% better end-to-end throughput, from a routing policy rather than more hardware.

Source: [Faster time-to-first-token with custom routing](https://www.anyscale.com/blog/ray-serve-faster-first-token-custom-routing), which also plots time per output token and per-replica throughput across the same sweep.

<div class="alert alert-block alert-info">
<code>code/llm/routing/verify.py</code> reproduces the comparison on two GPUs and asserts that affinity wins, so it also serves as a regression check on the routing config.
</div>

## 2. Prefill/decode disaggregation

Routing decided *which replica* serves a request. Disaggregation splits *one request's two phases* across separate pools, because one engine cannot independently hit both a TTFT and an inter-token-latency (ITL) target. Define the trade-off carefully, then the architecture, then when it pays.

### 2.1 Why one engine cannot win on both TTFT and ITL

The two phases sit on opposite sides of the roofline ridge, each with its own latency target, so one shared budget can only compromise between them.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/roofline_recap.png" loading="lazy" width="860">

- **Prefill is compute-bound and sets TTFT**
    - It processes the whole prompt in one pass, doing a lot of math per byte moved.
- **Decode is memory-bandwidth-bound and sets ITL**
    - Each step re-reads the weights plus the whole growing KV cache to emit one token.
    - The KV term dominates at long sequence.
- **One engine, one budget**
    - A co-located engine serves both phases from one per-step token budget, `max_num_batched_tokens`.
    - Every prompt token scheduled for prefill is a slice not spent advancing live decodes.

Worked trade-off (numbers illustrative): 32 users streaming, each expecting a token roughly every 25 ms, when a new request arrives with an 8,000-token prompt.

- **Tune for TTFT** — `max_num_batched_tokens=8192`
    - Prefill the whole prompt in about one 300 ms step, so TTFT is great (~300 ms).
    - But the 32 streams get almost no decode steps for those 300 ms: ITL spikes from 25 ms to ~300 ms, about 12x.
    - Everyone stutters.
- **Tune for ITL** — `max_num_batched_tokens=512`, prioritizing decode
    - Each step spends most of the budget on the 32 decodes, so ITL stays smooth at ~25 ms.
    - That leaves only ~480 prompt tokens per step (512 budget minus 32 decode tokens).
    - The 8,000-token prefill spreads over ~17 steps, so the new user's TTFT climbs to ~1.5-2 s.
- **Takeaway**
    - Lower the budget and you protect ITL but raise TTFT; raise it and you cut TTFT but spike ITL.
    - You would rather scale prefill and decode independently, which one engine cannot do.

### 2.2 The disaggregated architecture

Split the two phases onto separate worker pools so each is sized to its own bottleneck instead of time-slicing one budget.

- **Decode orchestrates**
    - The request path is `ingress -> PDDecodeServer -> PDPrefillServer`.
    - The decode server requests a prefill, receives the prompt's KV, then streams tokens.
- **KV transfer**
    - The prompt's KV moves from prefill VRAM to decode VRAM over NIXL, a point-to-point GPU-to-GPU transport.
    - It completes before the first token, so the transfer lands inside TTFT.
    - Configured with `kv_transfer_config`.
- **One model, two pools**
    - Both pools serve the SAME `model_id`.
    - The builder enforces matching ids and a `kv_transfer_config` on each.

The decode server orchestrates: it requests a prefill, the KV cache flows back over NIXL, and decode streams the tokens. The two pools are sized independently, each tagged with its roofline regime.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/prefill_decode_disagg.png" loading="lazy" width="1000">

### 2.3 Independent sizing, and when it pays

Each pool gets its own replica count, which is the lever section 2.1 wanted: add prefill capacity to cut TTFT, add decode capacity to raise throughput and protect ITL, without touching the other.

- **The independent lever**
    - "Scale only prefill to improve TTFT" is now real.
    - It trades against aggregate throughput: GPUs spent on prefill are not decoding.
- **When it pays**
    - Distinct TTFT and ITL targets, high traffic, and enough GPUs to dedicate pools.
    - Long prompts, so prefill is expensive enough to be worth isolating.
    - Long outputs, so the one-time KV transfer is repaid across many decode steps.
    - A fast fabric between the pools: NVLink within a node, InfiniBand or RoCE across nodes.
- **When it does not**
    - Below that, one co-located engine with chunked prefill is simpler.
    - Short prompts and short replies, where the transfer can cost more than the split saves.

The disaggregated app is two `LLMConfig`s, one per phase, each carrying a `kv_transfer_config`:

```python
from ray.serve.llm import LLMConfig, build_pd_openai_app   # PublicAPI(stability="alpha") on Ray 2.56

def phase_config() -> LLMConfig:                     # prefill and decode share this shape
    return LLMConfig(
        model_loading_config=dict(model_id="qwen-0.5b",   # SAME model_id for both pools
                                  model_source=MODEL_SOURCE),
        runtime_env=MODEL_ENV,
        engine_kwargs=dict(
            load_format="runai_streamer",
            max_model_len=4096,
            kv_transfer_config=dict(kv_connector="NixlConnector", kv_role="kv_both"),  # mandatory KV transport
        ),
    )

prefill = phase_config()
prefill.deployment_config = dict(autoscaling_config=dict(min_replicas=1, max_replicas=4))  # compute pool: N
decode = phase_config()
decode.deployment_config = dict(autoscaling_config=dict(min_replicas=2, max_replicas=8))   # decode pool: M

app = build_pd_openai_app(dict(prefill_config=prefill, decode_config=decode))
```

Note: both pools use the same `model_id` (one model, split by phase) and both set `kv_transfer_config`; the builder enforces this. Pools size independently: `prefill` replicas for TTFT, `decode` replicas for throughput and ITL.

## 3. Mixture-of-experts at scale

This is where spreading gets built. The section defines what a mixture of experts is, shards the part of it that dominates memory, replicates the part that does not, then composes the three knobs that express both.

### 3.1 MoE primer

A decoder layer is attention plus a feed-forward block, and MoE swaps only the feed-forward half for a gate-and-experts version.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/moe_layer_anatomy.png" loading="lazy" width="960">

Dense versus mixture-of-experts, defined before any parallelism.

- **Expert** = one feed-forward network inside a layer
    - An MoE layer holds many: for example 256 routed experts in a DeepSeek-V3-class model.
- **Gate (router)** = a small learned function that picks which experts fire
    - It scores the experts for each token and keeps the top-k, for example top-8 of 256.
    - The rest stay idle for that token.
- **Params huge, active FLOPs small**
    - Total parameters scale with the expert count.
    - Active FLOPs per token scale only with top-k.
    - DeepSeek-V3 is **671B parameters, of which 37B are active per token**: 256 experts per layer at 44M each, top-8 fired.
- **MoE swaps only the feed-forward HALF of a layer**
    - A layer is attention (mixes tokens, uses the KV cache) plus a feed-forward block (transforms each token alone).
    - MoE replaces only the feed-forward block. Attention is untouched.
- **The experts are the model**
    - Which is why the parallelism that follows shards them and replicates attention, not the reverse.

A dense layer already keeps most of its parameters in the feed-forward block. Swapping that block for 256 copies of it makes the share total.

| | attention | feed-forward / experts |
|---|---:|---:|
| Llama-3-70B, per layer | 151 M | 705 M — **82%** |
| Qwen2.5-7B, per layer | 29 M | 204 M — **87%** |
| DeepSeek-V3, whole model | 11.4 B — 2% | 654 B — **97%** |

Dense sends every token through one feed-forward network; MoE routes each token to a small subset of experts, so the parameters are huge but the active FLOPs per token stay small.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/moe_vs_dense.png" loading="lazy" width="960">

### 3.2 Expert parallelism: shard what dominates

The same GPUs seen two ways: attention replicated per rank, experts split across all of them, with all-to-all bridging the two layouts every MoE layer.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/dp_ep_two_views_v2.png" loading="lazy" width="1000">

- **Expert parallelism (EP)** = shard a layer's experts across GPUs, each holding a slice
    - A token routes to the ranks owning its top-k experts, and the outputs come back.
    - Attention is untouched.
- **Why not tensor parallel for the experts**
    - It slices every expert matrix across every rank, so each does a sliver of every token's work.
    - EP keeps each expert whole and lets the gate's sparsity decide which ranks work.
- **all-to-all** = a distributed transpose: each rank sends a different slice to every other rank, nothing replicated.

One MoE layer across two ranks: attention and routing stay local, only the experts make tokens travel through dispatch and combine.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/moe_layer_walkthrough.png" loading="lazy" width="1000">

- **Volume per layer** = `batch_tokens x top_k x hidden x dtype_bytes`, each way
    - DeepSeek-V3 at a batch of 1,024 tokens: **13.6 GB per forward pass** over its 58 MoE layers.
    - Hence NVLink or InfiniBand, and a low-latency all-to-all kernel rather than a stock collective.
- **`top_k` is the only term that grows it**
    - Per-GPU memory falls as you widen while the communication bill stays flat.
    - That is the argument for wide EP.

The other half of the layout: what keeps every one of those shards fed.

- **Data-parallel attention** = replicate the attention path on every rank
    - Each rank serves its own request stream, so a sequence's KV lives in exactly one place.
- **It is what feeds the shards**
    - Experts over 64 GPUs is 64 shards wanting tokens every step, and one stream cannot supply them.
    - Expert parallelism without enough data parallelism is idle shards.
- **Every rank joins both collectives, so the slowest sets the pace**
    - Uneven per-rank load becomes idle GPU time, which is why the scheduler balances tokens, not requests.
- **It buys throughput, not single-stream latency.** Tensor parallel makes one request faster; data parallel runs it on one rank.

Sharding the experts is one boolean; the width is not a setting at all, just `data_parallel_size * tensor_parallel_size`.

```python
engine_kwargs = dict(
    enable_expert_parallel=True,    # shard experts across the deployment's GPUs
    data_parallel_size=16,          # 16 attention replicas keep those shards fed
)
```

<div class="alert alert-block alert-info">
Gating is learned, so a hot expert makes its rank the straggler for the whole group. vLLM ships an expert-parallelism load balancer (<code>enable_eplb</code>) that rebalances placement at runtime, set through <code>engine_kwargs</code> like any other engine knob.
</div>

### 3.3 Composing the three knobs

Dense models scale by **slicing** one copy: tensor parallel inside a node, pipeline parallel across nodes. Mixture-of-experts models scale by **spreading**. A frontier deployment sets all three knobs, because each targets a different bottleneck.

| knob | what it shards | what it costs |
|---|---|---|
| `tensor_parallel_size` | attention weights and the dense feed-forward layers, inside each DP rank | an all-reduce per layer |
| `data_parallel_size` | nothing: replicates the attention path, one request stream per rank | replicated attention weights, and lockstep at every MoE collective |
| `enable_expert_parallel` | the experts, across all `dp x tp` GPUs, **instead of** tensor parallel | two all-to-alls per MoE layer |

Spreading DeepSeek-V3's experts wider shrinks the sharded term while the replicated one holds still.

| experts spread over | expert weights / GPU | replicated attention | left of an 80 GB GPU |
|---:|---:|---:|---:|
| 8 GPUs | 164 GB | 23 GB | does not fit |
| 16 GPUs | 82 GB | 23 GB | does not fit |
| 32 GPUs | 41 GB | 23 GB | 16 GB |
| 64 GPUs | 20 GB | 23 GB | 37 GB |

- **Pick the narrowest width that fits and still leaves KV room**, not a number off a menu.
- **The middle column is a floor**, and tensor parallel is how you cut it
    - Past about 32 GPUs the replicated attention is the larger term, and widening stops paying.
    - `tensor_parallel_size=8` brings that 23 GB under 3 GB, so more room for KV cache.
- **Tensor parallel does double duty**
    - Expert-layer width is `dp x tp`.
    - So raising it shrinks the attention floor *and* widens the expert spread at once.
- **Read section 4's `data_parallel_size: 16, tensor_parallel_size: 8` with expert parallel on as**
    - Experts spread over 128 GPUs.
    - Attention sharded 8 ways inside each of 16 request streams.

### 3.4 The DP group: config and lifecycle

One DP group is one logical copy: each rank holds a replicated attention path and a different expert shard, coupled by all-to-all, so all ranks are required for the collective.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/dp_group.png" loading="lazy" width="900">

- **It is an MoE-only feature, so this config is a shape rather than a run**
    - vLLM accepts `data_parallel_size > 1` only on an MoE model, and Qwen-0.5B is dense.
    - On a dense model it raises `Non-MoE models do not support external data parallel mode`.
    - Dense "data parallelism" is just independent replicas, which section 1.4 already showed.
- **`build_dp_openai_app` takes a SINGULAR `llm_config`**
    - Not the `llm_configs` list `build_openai_app` takes.
    - Mixing the two up is the common error.
- **GPU accounting**
    - `num_replicas` is the number of DP groups.
    - Total GPUs = `num_replicas * data_parallel_size * tensor_parallel_size`.

```python
dp_config = LLMConfig(
    model_loading_config=dict(model_id="qwen-0.5b", model_source=MODEL_SOURCE),
    deployment_config=dict(autoscaling_config=dict(min_replicas=1, max_replicas=1)),  # one DP group
    runtime_env=MODEL_ENV,
    engine_kwargs=dict(
        load_format="runai_streamer",
        data_parallel_size=2,        # two ranks = one DP group of two GPUs
        tensor_parallel_size=1,      # no intra-rank sharding for a 0.5B model
        max_model_len=4096,
        enforce_eager=True,
        # enable_expert_parallel=True,   # add on a real MoE model; omitted here (Qwen-0.5B is dense)
    ),
)
```

Note: on an MoE model, `build_dp_openai_app({"llm_config": dp_config})` serves this as one DP group. Serve gang-schedules the two ranks and packs them onto the fewest nodes automatically.

Because the ranks are coupled by collectives, the group recovers as a unit rather than one replica at a time.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/dp_fault_tolerance.png" loading="lazy" width="1000">

- **Gang recovery**
    - One dead rank stalls the group's all-to-all.
    - Serve marks the whole group unhealthy, stops routing to it, reschedules the gang atomically, then resumes once healthy.
    - You cannot restart just one rank.
- **Autoscaling in DP increments.** One more group is one more gang of `data_parallel_size` ranks.
- **Automatic.** Any deployment with `data_parallel_size > 1` gets this gang lifecycle, no config.

## 4. Tier-2 multi-node production configs

Same builders, bigger gang: only the model, accelerator, parallel size, and node count change between the runnable 2-GPU config and the multi-node production config.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/config_diff.png" loading="lazy" width="1000">

<div class="alert alert-block alert-info">
<b>Naming the shape.</b> A deployment is described in two layers: the prefill/decode split as <code>xPyD</code> (x prefill instances, y decode instances), and each pool's per-instance parallelism as data-parallel size times tensor-parallel size with expert parallelism on or off. The first config below starts at <code>1P2D</code> and autoscales to <code>4P8D</code>; the second is one pool at <code>data_parallel_size 16</code> times <code>tensor_parallel_size 8</code> with expert parallelism on.
</div>

Prefill/decode disaggregation on a multi-node NIXL-capable cluster. `accelerator_type` keeps the config chip-agnostic; the model is a Kimi-class MoE checkpoint, used as a neutral example:

```yaml
# disaggregation/service_prod.yaml
name: pd-disagg-prod
image_uri: anyscale/ray-llm:2.56.0-py312-cu130    # must include the NIXL connector
applications:
  - name: pd-disagg-prod
    import_path: ray.serve.llm:build_pd_openai_app   # the SAME builder as the Tier-1 app
    args:
      prefill_config:                                # compute pool: fewer replicas, big token budget
        model_loading_config: {model_id: moe-chat, model_source: moonshotai/Kimi-K2-Instruct}
        accelerator_type: H100
        deployment_config: {autoscaling_config: {min_replicas: 1, max_replicas: 4}}
        engine_kwargs:
          pipeline_parallel_size: 2   # 2 nodes per replica: a ~1TB checkpoint exceeds one 8xH100 node (640 GB)
          tensor_parallel_size: 8     # shard each rank across the 8 GPUs in a node
          max_model_len: 131072
          kv_transfer_config: {kv_connector: NixlConnector, kv_role: kv_both}
      decode_config:                                 # decode pool: more replicas, steady-state bottleneck
        model_loading_config: {model_id: moe-chat, model_source: moonshotai/Kimi-K2-Instruct}   # SAME model_id
        accelerator_type: H100
        deployment_config: {autoscaling_config: {min_replicas: 2, max_replicas: 8}}
        engine_kwargs:
          pipeline_parallel_size: 2
          tensor_parallel_size: 8
          max_model_len: 131072
          kv_transfer_config: {kv_connector: NixlConnector, kv_role: kv_both}
```

Wide expert parallelism plus data-parallel attention, multi-node:

```yaml
# expert_parallel/service_prod.yaml
name: wideep-prod
image_uri: anyscale/ray-llm:2.56.0-py312-cu130
applications:
  - name: wideep-prod
    import_path: ray.serve.llm:build_dp_openai_app   # the SAME builder as the Tier-1 app
    args:
      llm_config:                                    # SINGULAR (not llm_configs)
        model_loading_config:
          model_id: moe-wide
          model_source: deepseek-ai/DeepSeek-V3      # a DeepSeek-class MoE checkpoint
        accelerator_type: H100
        engine_kwargs:
          data_parallel_size: 16                     # 16-way data-parallel attention
          tensor_parallel_size: 8                    # shard each rank across 8 GPUs
          enable_expert_parallel: true               # wide expert parallelism: shard the experts
          max_model_len: 65536
        # Serve gang-schedules the 16 ranks; each rank needs a full 8-GPU node, so the gang fans across 16 nodes.
```

<div class="alert alert-block alert-info">
<b>Reference implementations:</b> KV-cache-aware routing is <code>code/llm/routing/</code> (<code>cd code/llm/routing && serve run app:build_app</code>); prefill/decode disaggregation is <code>code/llm/disaggregation/</code>; data-parallel attention with wide expert parallelism is <code>code/llm/expert_parallel/</code>. Each holds an <code>app.py</code> with its paired <code>service.yaml</code> or Tier-2 <code>service_prod.yaml</code>.
</div>